In [0]:
%sql
USE CATALOG ecommerce_db;
CREATE OR REPLACE VIEW gold.category_daily AS
SELECT category_code,to_date(event_time) AS event_date,
COUNT(DISTINCT CASE WHEN event_type = 'view' THEN user_id else NULL end) as views,
COUNT(DISTINCT CASE WHEN event_type = 'cart' THEN user_id else NULL end) as carts,
COUNT(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id else NULL end) as purchases
FROM silver.events
where category_code IS NOT NULL
GROUP BY category_code,to_date(event_time)

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [0]:
# Prepare data
df = spark.table("gold.category_daily").toPandas()
X = df[["views", "carts"]]
y = df["purchases"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [0]:
# MLflow experiment
with mlflow.start_run(run_name="linear_regression_v1"):
    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)

    # Train
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluate
    score = model.score(X_test, y_test)
    mlflow.log_metric("r2_score", score)

    # Log model
    mlflow.sklearn.log_model(model, "model")

print(f"R² Score: {score:.4f}")


2026/01/20 11:42:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


R² Score: 0.9462


In [0]:
# Prepare data
df = spark.table("gold.category_daily").toPandas()
X = df[["views", "carts"]]
y = df["purchases"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4)

In [0]:
# MLflow experiment
with mlflow.start_run(run_name="linear_regression_v2"):
    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.4)

    # Train
    model1 = LinearRegression()
    model1.fit(X_train, y_train)

    # Evaluate
    score = model1.score(X_test, y_test)
    mlflow.log_metric("r2_score", score)

    # Log model
    mlflow.sklearn.log_model(model, "model1")

print(f"R² Score: {score:.4f}")


2026/01/20 11:47:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


R² Score: 0.7019


#### 📊  Interpreting These Two Models (Using R² Score)

**Two trained models**, and comparing them using **R² (R-squared) score**.

####🔹 What R² Score Means 

**R² tells you how well your model explains the data.**

* R² = **1.0** → Perfect prediction
* R² = **0.0** → Model is no better than guessing the average
* Higher R² → Better model fit
👉 You can think of R² as **“how much of the data’s behavior my model understands.”**

---

### 🧠 Model Comparison

#### ✅ Model 1

**R² ≈ 0.946**
* Explains **~94.6% of the variance** in the data
* Very strong fit
* Predictions are very close to actual values

👉 **This is a high-performing model**

---

#### ⚠️ **Model 2**

**R² ≈ 0.702**
* Explains **~70.2% of the variance**
* Decent but noticeably weaker than Model 1
* More unexplained error compared to Model 1

👉 **This model is acceptable but not the best**

---

#### 🏆 Final Interpretation

| Model           | R² Score    | Interpretation           |
| --------------- | ----------- | ------------------------ |
| Model 1         | ~0.946      | Excellent performance    |
| Model 2         | ~0.702      | Moderate performance     |
| **Best Choice** | **Model 1** | More accurate & reliable |


